# Agent Repair Pipeline — 2WikiMultiHopQA

**ICLR Experiment Pipeline — Colab A100-80GB + Qwen2.5-32B-Instruct-AWQ**

| Stage | What it does | ~Time (A100-80GB, 500 Qs) |
|---|---|---|
| 0 | Download 2WikiMultiHopQA, create folders | 1 min |
| 1 | ReAct trajectory generation (Qwen2.5-32B-Instruct-AWQ) | ~1-2 h |
| 2 | Step-level uncertainty scoring | ~1-2 h |
| 3 | 72B judge error annotation | ~1-2 h |
| 4 | Localization scoring (+ cascade-aware rules) | < 1 min |
| 5 | 126 repair strategies (backtrack × nudge type ablation) | ~4-6 h |
| 6 | Tables, stats, figures, ablation analysis | < 1 min |

**Every stage is resumable** — if Colab disconnects, re-run the cell and it picks up where it left off.

---
## 0. Setup

In [1]:
# Verify GPU — should show A100
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

/bin/bash: line 1: nvidia-smi: command not found


In [1]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

REPO_DIR = '/content/agent-repair'
CONFIG = 'config/config_colab_2wikimultihopqa.yaml'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/kishormorol/agent-repair.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

Cloning into '/content/agent-repair'...
remote: Enumerating objects: 333, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 333 (delta 9), reused 3 (delta 1), pack-reused 313 (from 1)
Receiving objects: 100% (333/333), 4.86 MiB | 20.49 MiB/s, done.
Resolving deltas: 100% (168/168), done.
Working directory: /content/agent-repair


In [3]:
# Check Colab's CUDA version first
!nvcc --version 2>/dev/null || echo "nvcc not found"
!python -c "import torch; print(f'torch CUDA: {torch.version.cuda}')" 2>/dev/null || echo "torch not installed yet"
!nvidia-smi | grep "CUDA Version"

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
torch CUDA: 12.8
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |


In [4]:
# Install vLLM — pinned to 0.19.0 (last version with CUDA 12 default wheels)
# vLLM >= 0.20 ships CUDA 13 wheels which don't work on Colab's CUDA 12.x
!pip uninstall -y vllm 2>/dev/null
!pip install -q "vllm==0.19.0" 2>&1 | tail -5

# Install remaining dependencies
!pip install -q "transformers>=4.57.0" "tokenizers>=0.21.0" accelerate \
    huggingface_hub datasets scipy scikit-learn pandas pyarrow \
    pyyaml tqdm matplotlib seaborn statsmodels 2>&1 | tail -3

print('\n--- Installation complete ---')

google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
google-adk 2.4.0 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.4.0 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.44.0 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.

--- Installation complete ---


In [5]:
# Quick sanity check: can vLLM see the GPU?
import torch, vllm
print(f'torch:  {torch.__version__}')
print(f'vLLM:   {vllm.__version__}')
print(f'CUDA:   {torch.version.cuda}')
print(f'GPU:    {torch.cuda.get_device_name(0)}')
print(f'VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

torch:  2.10.0+cu128
vLLM:   0.19.0
CUDA:   12.8
GPU:    NVIDIA A100-SXM4-80GB
VRAM:   85.1 GB


In [6]:
# Clear previous data
import shutil, os
drive_base = '/content/drive/MyDrive/agent-repair-2wikimultihopqa'
for d in ['outputs', 'data/processed']:
    p = os.path.join(drive_base, d)
    if os.path.exists(p):
        shutil.rmtree(p)
        print(f'Cleared {p}')
for d in ['outputs', 'data/processed']:
    p = os.path.join('/content/agent-repair', d)
    if os.path.exists(p):
        shutil.rmtree(p)
        print(f'Cleared {p}')
print('Ready for fresh run with 2WikiMultiHopQA')

Ready for fresh run with 2WikiMultiHopQA


---
## Smoke Test (20 questions, ~10-15 min)

**Run this first** to verify everything works before committing A100 units to the full 500-question run.

In [7]:
stages = ['run_setup', 'run_generate', 'run_uncertainty', 'run_annotate',
          'run_localize', 'run_repair', 'run_eval']

for stage in stages:
    print(f'\n{"="*60}\n  {stage}\n{"="*60}')
    !python scripts/{stage}.py --config {CONFIG} --limit 20
    print()


  run_setup
03:56:20 | INFO    | setup | config=config/config_colab_2wikimultihopqa.yaml  base=/content/drive/MyDrive/agent-repair-2wikimultihopqa
03:56:20 | INFO    | setup | Dataset: 2wikimultihopqa (file: 2wikimultihop_dev.json)
03:56:20 | INFO    | setup | Downloading 2wikimultihopqa via registry download function...
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'framolfese/2WikiMultihopQA' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
README.md: 5.46kB [00:00, 22.0MB/s]
data/train-00000-of-00002.parquet: 100% 166M/166M [00:03<00:00, 43.3MB/s]
data/train-00001-of-00002.parquet: 100% 165M/165M [00:03<00:00, 43.5MB/s]
data/validation-00000-of-00001.parquet: 100% 29.5M/29.5M [00:01<00:00, 27.2MB/s]
data/test-00000-of-00001.parquet: 100% 28.0M/28.0M [00:00<00:00, 28.8MB/s]
Generating train

In [8]:
# Check smoke test produced output
import glob
tables = glob.glob('outputs/tables/*') + glob.glob('/content/drive/MyDrive/agent-repair-2wikimultihopqa/outputs/tables/*')
figs = glob.glob('outputs/figures/*') + glob.glob('/content/drive/MyDrive/agent-repair-2wikimultihopqa/outputs/figures/*')
print(f'Tables: {len(tables)}, Figures: {len(figs)}')
if tables or figs:
    print('Smoke test PASSED — safe to run the full pipeline below.')
else:
    print('WARNING: No outputs found. Check the logs above for errors.')

Tables: 4, Figures: 0
Smoke test PASSED — safe to run the full pipeline below.


---
## Full Pipeline (500 questions)

Only run these cells after the smoke test passes. To reset and run fresh, delete the `data/processed/` folder.

### Stage 0: Download 2WikiMultiHopQA

In [9]:
!python scripts/run_setup.py --config {CONFIG}

04:12:35 | INFO    | setup | config=config/config_colab_2wikimultihopqa.yaml  base=/content/drive/MyDrive/agent-repair-2wikimultihopqa
04:12:35 | INFO    | setup | Dataset: 2wikimultihopqa (file: 2wikimultihop_dev.json)
04:12:35 | INFO    | setup | Dataset already present: /content/drive/MyDrive/agent-repair-2wikimultihopqa/data/raw/2wikimultihop_dev.json
04:12:36 | INFO    | setup | Dataset ready: /content/drive/MyDrive/agent-repair-2wikimultihopqa/data/raw/2wikimultihop_dev.json (64.2 MB, 12576 questions)
04:12:36 | INFO    | setup | Output dirs created under: /content/drive/MyDrive/agent-repair-2wikimultihopqa


### Stage 1: Generate ReAct Trajectories

In [10]:
!python scripts/run_generate.py --config {CONFIG}

04:12:37 | INFO    | stage1 | config=config/config_colab_2wikimultihopqa.yaml  base=/content/drive/MyDrive/agent-repair-2wikimultihopqa
04:12:37 | INFO    | stage1 | Dataset: 2wikimultihopqa (env: WikiMultiHopEnv)
04:12:37 | INFO    | stage1 | Pool: 500 questions
04:12:37 | INFO    | stage1 | To process: 480 of 500 (20 already done)
04:12:39 | INFO    | stage1 | GPU VRAM: 85.094825984 GB | agent model: Qwen/Qwen2.5-32B-Instruct-AWQ (85 GB -> fp16)
2026-07-31 04:12:42.326789: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-31 04:12:42.399998: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA,

### Stage 2: Step-Level Uncertainty

In [11]:
!python scripts/run_uncertainty.py --config {CONFIG}

04:18:52 | INFO    | stage2 | config=config/config_colab_2wikimultihopqa.yaml  base=/content/drive/MyDrive/agent-repair-2wikimultihopqa
04:18:52 | INFO    | stage2 | Math metrics: 480 of 500 trajectories to do
04:19:04 | INFO    | stage2 | Math metrics done.
04:19:04 | INFO    | stage2 | Sampling metrics: 194 of 203 failed trajectories
04:19:06 | INFO    | stage2 | GPU VRAM: 85.094825984 GB | agent model: Qwen/Qwen2.5-32B-Instruct-AWQ (85 GB -> fp16)
2026-07-31 04:19:09.537315: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-31 04:19:09.610038: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI F

### Stage 3: Error Annotation (72B Judge)

In [12]:
!python scripts/run_annotate.py --config {CONFIG}

04:25:40 | INFO    | stage3 | config=config/config_colab_2wikimultihopqa.yaml  base=/content/drive/MyDrive/agent-repair-2wikimultihopqa
04:25:40 | INFO    | stage3 | To annotate: 194 of 203 failed trajectories
04:25:42 | INFO    | stage3 | GPU VRAM: 85.094825984 GB | judge: Qwen/Qwen2.5-72B-Instruct-AWQ (85 GB -> 72B judge)
2026-07-31 04:25:45.457656: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-31 04:25:45.529250: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
INFO 07-31 04:25:56 [utils.py:233] non-default a

### Stage 4: Localization Scoring

In [13]:
!python scripts/run_localize.py --config {CONFIG}

04:31:13 | INFO    | stage4 | config=config/config_colab_2wikimultihopqa.yaml  base=/content/drive/MyDrive/agent-repair-2wikimultihopqa
04:31:14 | INFO    | stage4 | records: 2030 (203 trajectories x 10 metrics)
                        argmax_top1  argmax_within1    mrr  threshold_hit  threshold_within1  top2_hit  top3_hit    n
metric                                                                                                               
self_consistency              0.404           0.591  0.581          0.335              0.557     0.562     0.700  203
token_entropy_max             0.246           0.512  0.486          0.276              0.591     0.453     0.660  203
max_token_prob_max            0.232           0.507  0.484          0.266              0.591     0.468     0.695  203
token_entropy_mean_k50        0.177           0.374  0.418          0.192              0.438     0.305     0.581  203
token_entropy_mean            0.177           0.374  0.418          0.192       

### Stage 5: Repair (126 Strategies)

In [14]:
!python scripts/run_repair.py --config {CONFIG}

04:31:14 | INFO    | stage5 | config=config/config_colab_2wikimultihopqa.yaml  base=/content/drive/MyDrive/agent-repair-2wikimultihopqa
04:31:14 | INFO    | stage5 | Dataset: 2wikimultihopqa (env: WikiMultiHopEnv)
04:31:14 | INFO    | stage5 | 203 failed x 126 strategies x 3 seeds = 76734 result rows (actual GPU runs are far fewer: dedup)
04:31:16 | INFO    | stage5 | GPU VRAM: 85.094825984 GB | agent model: Qwen/Qwen2.5-32B-Instruct-AWQ (85 GB -> fp16)
2026-07-31 04:31:19.646470: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-31 04:31:19.717656: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNN

### Stage 6: Evaluation

In [15]:
!python scripts/run_eval.py --config {CONFIG}

05:35:27 | INFO    | stage6 | config=config/config_colab_2wikimultihopqa.yaml  base=/content/drive/MyDrive/agent-repair-2wikimultihopqa
05:35:29 | INFO    | stage6 | rows: 77343 | uncertainty strategies: 120 (bt0: 60) | + ensemble

MAIN RESULTS  (fixed_% = share of FAILED trajectories that the repair fixed)
                                                           strategy  fixed_%     95%_CI  avg_tokens  avg_tool_calls  hit_true_step_%
                                                        random_step     19.2 16.3-22.3%         170            2.36             21.3
                                                       full_restart     20.8 17.7-24.1%         246            4.03              3.0
                                               oracle_targeted__bt2     22.3 19.1-25.8%         234            3.67              3.0
                                     oracle_targeted__bt2__informed     21.8 18.6-25.1%         231            3.63              3.0
                          

---
## Results

In [16]:
from src.utils import load_config
cfg = load_config(CONFIG)

summary_path = os.path.join(cfg.path('tables'), 'summary.txt')
if os.path.exists(summary_path):
    with open(summary_path) as f:
        print(f.read())
else:
    print('Summary not yet generated. Run Stage 6 first.')

Summary not yet generated. Run Stage 6 first.


In [17]:
import pandas as pd

results_path = os.path.join(cfg.path('tables'), 'main_results_readable.csv')
if os.path.exists(results_path):
    df = pd.read_csv(results_path)
    display(df)
else:
    print('Results table not yet generated. Run Stage 6 first.')

,strategy,fixed_%,95%_CI,avg_tokens,avg_tool_calls,hit_true_step_%
0,random_step,19.2,16.3-22.3%,170,2.36,21.3
1,full_restart,20.8,17.7-24.1%,246,4.03,3.0
2,oracle_targeted__bt2,22.3,19.1-25.8%,234,3.67,3.0
3,oracle_targeted__bt2__informed,21.8,18.6-25.1%,231,3.63,3.0
4,oracle_targeted__informed,12.3,9.8-14.9%,177,2.37,100.0
...,...,...,...,...,...,...
122,unc__verbalized_confidence__topk__bt2,25.4,22.0-28.9%,239,3.81,7.4
123,unc__verbalized_confidence__topk__bt2__informed,24.0,20.7-27.4%,236,3.81,7.4
124,unc__verbalized_confidence__topk__informed,24.5,21.2-27.9%,214,3.36,11.8
125,uncertainty_ensemble_any,47.0,43.0-50.9%,11423,166.80,89.7


In [18]:
import glob
from IPython.display import display, Image

fig_dir = cfg.path('figures')
figs = sorted(glob.glob(os.path.join(fig_dir, '*.png')))
if figs:
    for f in figs:
        print(f'\n--- {os.path.basename(f)} ---')
        display(Image(filename=f, width=800))
else:
    print('No figures yet. Run Stage 6 first.')

No figures yet. Run Stage 6 first.


---
## Re-run Stages 4–6 with Cascade-Aware Strategies

**Use this after you already have stages 1–3 completed** (trajectories, uncertainty, annotations).
This pulls the latest code (with cascade rules), clears only localization/repair/eval outputs,
and re-runs stages 4→5→6 in one cell. ~3-4 hours on A100-80GB for 500 questions.

In [19]:
import os, shutil

REPO_DIR = '/content/agent-repair'
CONFIG = 'config/config_colab_2wikimultihopqa.yaml'

# 1. Pull latest code with cascade-aware strategies
os.chdir(REPO_DIR)
!git pull

# 2. Clear ONLY stages 4-6 outputs (keep trajectories, uncertainty, annotations)
drive_base = '/content/drive/MyDrive/agent-repair-2wikimultihopqa'
for base in [REPO_DIR, drive_base]:
    for d in ['outputs/localization', 'outputs/repairs', 'outputs/tables', 'outputs/figures']:
        p = os.path.join(base, d)
        if os.path.exists(p):
            shutil.rmtree(p)
            print(f'Cleared {p}')

# 3. Clear checkpoint files for stages 4-6 (these live in outputs/logs/)
for base in [REPO_DIR, drive_base]:
    for ckpt in ['outputs/logs/stage4_localize.jsonl',
                 'outputs/logs/stage5_repair.jsonl',
                 'outputs/logs/stage6_eval.jsonl']:
        p = os.path.join(base, ckpt)
        if os.path.exists(p):
            os.remove(p)
            print(f'Removed checkpoint: {p}')

print('\nStages 1-3 data preserved. Ready to re-run 4→5→6.')

Already up to date.
Cleared /content/drive/MyDrive/agent-repair-2wikimultihopqa/outputs/localization
Cleared /content/drive/MyDrive/agent-repair-2wikimultihopqa/outputs/repairs
Cleared /content/drive/MyDrive/agent-repair-2wikimultihopqa/outputs/tables
Cleared /content/drive/MyDrive/agent-repair-2wikimultihopqa/outputs/figures
Removed checkpoint: /content/drive/MyDrive/agent-repair-2wikimultihopqa/outputs/logs/stage5_repair.jsonl

Stages 1-3 data preserved. Ready to re-run 4→5→6.


In [20]:
# Run stages 4 → 5 → 6 sequentially (run this and go to sleep)
import time

for stage in ['run_localize', 'run_repair', 'run_eval']:
    t0 = time.time()
    print(f'\n{"="*60}\n  {stage}\n{"="*60}')
    !python scripts/{stage}.py --config {CONFIG}
    elapsed = (time.time() - t0) / 60
    print(f'  ✓ {stage} done in {elapsed:.1f} min\n')

print('\n' + '='*60)
print('  ALL DONE — check results below')
print('='*60)


  run_localize
05:35:38 | INFO    | stage4 | config=config/config_colab_2wikimultihopqa.yaml  base=/content/drive/MyDrive/agent-repair-2wikimultihopqa
05:35:39 | INFO    | stage4 | records: 2030 (203 trajectories x 10 metrics)
                        argmax_top1  argmax_within1    mrr  threshold_hit  threshold_within1  top2_hit  top3_hit    n
metric                                                                                                               
self_consistency              0.404           0.591  0.581          0.335              0.557     0.562     0.700  203
token_entropy_max             0.246           0.512  0.486          0.276              0.591     0.453     0.660  203
max_token_prob_max            0.232           0.507  0.484          0.266              0.591     0.468     0.695  203
token_entropy_mean_k50        0.177           0.374  0.418          0.192              0.438     0.305     0.581  203
token_entropy_mean            0.177           0.374  0.418      

In [21]:
# View cascade-aware results
from src.utils import load_config
cfg = load_config(CONFIG)

summary_path = os.path.join(cfg.path('tables'), 'summary.txt')
if os.path.exists(summary_path):
    with open(summary_path) as f:
        print(f.read())

import pandas as pd
results_path = os.path.join(cfg.path('tables'), 'main_results_readable.csv')
if os.path.exists(results_path):
    df = pd.read_csv(results_path)
    display(df)

import glob
from IPython.display import display, Image
figs = sorted(glob.glob(os.path.join(cfg.path('figures'), '*.png')))
for f in figs:
    print(f'\n--- {os.path.basename(f)} ---')
    display(Image(filename=f, width=800))

,strategy,fixed_%,95%_CI,avg_tokens,avg_tool_calls,hit_true_step_%
0,random_step,19.0,15.9-22.2%,170,2.36,21.3
1,full_restart,21.2,18.1-24.5%,246,4.03,3.0
2,oracle_targeted__bt2,22.0,18.7-25.3%,234,3.68,3.0
3,oracle_targeted__bt2__informed,21.0,17.9-24.3%,230,3.63,3.0
4,oracle_targeted__informed,12.2,9.5-14.8%,177,2.37,100.0
...,...,...,...,...,...,...
122,unc__verbalized_confidence__topk__bt2,24.6,21.2-28.1%,239,3.82,7.4
123,unc__verbalized_confidence__topk__bt2__informed,23.6,20.4-27.1%,236,3.82,7.4
124,unc__verbalized_confidence__topk__informed,24.3,21.0-27.8%,213,3.36,11.8
125,uncertainty_ensemble_any,46.0,42.0-49.9%,11426,166.56,89.7
